# Week 5: Model Training, Cost-Sensitive Evaluation & Bootstrapped CIs
## Fraud Risk Analytics & Detection System

> **Evaluation Principles:**
> 1. **Temporal Partition Isolation:** Models are trained strictly on $N=472,432$ ($TransactionDT \le 12,192,854$) and evaluated on held-out test data ($N=118,108$, $TransactionDT > 12,192,854$).
> 2. **Imbalance Metrics:** Under a $3.5\%$ fraud rate, **PR-AUC** and **Recall @ Fixed FPR (1%, 5%)** are the primary evaluation metrics.
> 3. **Statistical Uncertainty:** Every headline test metric is validated using **1,000-resample non-parametric Bootstrap 95% Confidence Intervals**.
> 4. **Generalization Lower Bound:** Model decay is audited against brand-new unseen entities (Grouped Entity Split benchmark).

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, roc_curve

# Setup paths
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.evaluation import (
    compute_classification_metrics,
    calculate_recall_at_fixed_fpr,
    bootstrap_metric_confidence_intervals,
)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = "Arial"
print("Modeling environment ready.")

## 1. Load Model Metrics & Evaluation Manifest

In [ ]:
metrics_path = PROJECT_ROOT / "models" / "model_metrics.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

champ = metrics["champion_model"]
base = metrics["baseline_model"]
cis = champ["bootstrapped_95_ci_1000_resamples"]

print(f"Champion Model:      {champ['name']}")
print(f"PR-AUC:              {champ['test_metrics']['pr_auc']:.4f} (95% CI: [{cis['pr_auc']['ci_95_low']:.4f}, {cis['pr_auc']['ci_95_high']:.4f}])")
print(f"ROC-AUC:             {champ['test_metrics']['roc_auc']:.4f} (95% CI: [{cis['roc_auc']['ci_95_low']:.4f}, {cis['roc_auc']['ci_95_high']:.4f}])")
print(f"Recall @ 1% FPR:     {champ['recall_at_1pct_fpr']['recall']*100:.2f}% (95% CI: [{cis['recall_at_1pct_fpr']['ci_95_low']*100:.2f}%, {cis['recall_at_1pct_fpr']['ci_95_high']*100:.2f}%])")
print(f"Recall @ 5% FPR:     {champ['recall_at_5pct_fpr']['recall']*100:.2f}% (95% CI: [{cis['recall_at_5pct_fpr']['ci_95_low']*100:.2f}%, {cis['recall_at_5pct_fpr']['ci_95_high']*100:.2f}%])")

## 2. Head-to-Head Comparison: Baseline (Logistic Regression) vs. Champion (LightGBM)

In [ ]:
comp_df = pd.DataFrame({
    "Metric": ["PR-AUC", "ROC-AUC", "Recall @ 1% FPR", "Recall @ 5% FPR"],
    "Baseline (Logistic Regression)": [
        f"{base['test_metrics']['pr_auc']:.4f}",
        f"{base['test_metrics']['roc_auc']:.4f}",
        f"{base['recall_at_1pct_fpr']['recall']*100:.2f}%",
        f"{base['recall_at_5pct_fpr']['recall']*100:.2f}%",
    ],
    "Champion (LightGBM)": [
        f"{champ['test_metrics']['pr_auc']:.4f}",
        f"{champ['test_metrics']['roc_auc']:.4f}",
        f"{champ['recall_at_1pct_fpr']['recall']*100:.2f}%",
        f"{champ['recall_at_5pct_fpr']['recall']*100:.2f}%",
    ],
    "95% Bootstrap Confidence Interval": [
        f"[{cis['pr_auc']['ci_95_low']:.4f} – {cis['pr_auc']['ci_95_high']:.4f}]",
        f"[{cis['roc_auc']['ci_95_low']:.4f} – {cis['roc_auc']['ci_95_high']:.4f}]",
        f"[{cis['recall_at_1pct_fpr']['ci_95_low']*100:.2f}% – {cis['recall_at_1pct_fpr']['ci_95_high']*100:.2f}%]",
        f"[{cis['recall_at_5pct_fpr']['ci_95_low']*100:.2f}% – {cis['recall_at_5pct_fpr']['ci_95_high']*100:.2f}%]",
    ]
})

comp_df

## 3. Top 20 Feature Importances (Information Gain)

In [ ]:
imp_df = pd.DataFrame(metrics["top_feature_importances"]).head(20)

plt.figure(figsize=(12, 7))
sns.barplot(data=imp_df, x="importance_gain", y="feature", palette="viridis")
plt.title("Champion LightGBM: Top 20 Features by Total Gain Importance", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Total Split Gain (Predictive Power)", fontweight="bold")
plt.ylabel("Feature Name", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Generalization Lower Bound: Unseen Entity Stress Test (0% Overlap)

In [ ]:
unseen = metrics.get("unseen_entity_benchmark (0% overlap lower bound)", {})
if unseen:
    print(f"Unseen Entity Transactions (Test): {unseen['total_unseen_transactions']:,}")
    print(f"Unseen Entity Fraud Rate:          {unseen['unseen_fraud_rate_pct']}%")
    print(f"Unseen Entity PR-AUC:              {unseen['pr_auc']:.4f} (vs {champ['test_metrics']['pr_auc']:.4f} overall)")
    print(f"Unseen Entity ROC-AUC:             {unseen['roc_auc']:.4f} (vs {champ['test_metrics']['roc_auc']:.4f} overall)")
    print(f"Unseen Recall @ 1% FPR:            {unseen['recall_at_1pct_fpr']*100:.2f}%")
    print(f"Unseen Recall @ 5% FPR:            {unseen['recall_at_5pct_fpr']*100:.2f}%")

## 5. Operational Threshold Trade-Offs (Review Queue Sizing)

In [ ]:
sweep_df = pd.DataFrame(metrics["threshold_sweep_summary"])
sweep_df[["threshold", "recall", "precision", "fpr", "review_rate", "true_positives", "false_positives"]]